<a href="https://colab.research.google.com/github/sugandhasrivastava0610-blip/AI-Powered-Traffic-Signal-Controller/blob/main/Ai_traffic_signal_controller_optimized.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install ultralytics roboflow inference-sdk streamlit opencv-python-headless -q
print("✅ Done!")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.1/60.1 kB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 6.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 249.2/249.2 kB 20.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.9/49.9 MB 17.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.8/66.8 kB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 74.3/74.3 kB 6.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.2/9.2 MB 56.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.5/62.5 MB 10.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 49.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.1/7.1 MB 49.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.5/5.5 MB 62.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.3/11.3 MB 48.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 5.8 M

In [2]:
from roboflow import Roboflow
import glob

API_KEY = "FQDuUrf48Rcmxpql57Jb"  # ← get fresh key NOW

rf      = Roboflow(api_key=API_KEY)
project = rf.workspace("rjacaac1").project("ua-detrac-dataset-10k")
dataset = project.version(1).download("yolov8")

all_images = glob.glob(dataset.location + "/test/images/*.jpg")
if not all_images:
    all_images = glob.glob(dataset.location + "/valid/images/*.jpg")
if not all_images:
    all_images = glob.glob(dataset.location + "/train/images/*.jpg")

# Save paths to file so app.py can read them
with open("/content/dataset_path.txt","w") as f:
    f.write(dataset.location)

!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64
!chmod +x cloudflared-linux-amd64

print(f"✅ Dataset ready: {len(all_images)} images")
print("✅ Cloudflared ready!")

loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to UA-DETRAC-DATASET-10K-1 in yolov8:: 100%|██████████| 33380/33380 [00:09<00:00, 3560.40it/s]


Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
✅ Dataset ready: 487 images
✅ Cloudflared ready!


In [3]:
%%writefile app.py
import streamlit as st
import requests, cv2, tempfile, os, time, random, math, io
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from PIL import Image
from collections import Counter, deque
from io import BytesIO
import pandas as pd

try:
    from reportlab.lib.pagesizes import A4
    from reportlab.lib import colors
    from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
    from reportlab.platypus import SimpleDocTemplate, Paragraph, Spacer, Table, TableStyle
    from reportlab.lib.units import inch
    PDF_OK = True
except:
    PDF_OK = False

st.set_page_config(page_title="AI Traffic Signal Controller",
    page_icon="🚦", layout="wide", initial_sidebar_state="expanded")

API_KEY  = "FQDuUrf48Rcmxpql57Jb"
MODEL_ID = "ua-detrac-dataset-10k/1"
RF_URL   = "https://serverless.roboflow.com"
VEHICLE_WEIGHTS = {"car":1.0,"van":1.5,"bus":2.5,"truck":3.0}
MAX_WAIT        = 120
LANE_NAMES      = ["NORTH","SOUTH","EAST","WEST"]
LANE_ARROW      = {"NORTH":"N","SOUTH":"S","EAST":"E","WEST":"W"}
CLASS_COLORS    = {"car":"#4ade80","truck":"#f87171","bus":"#60a5fa","van":"#fbbf24"}
CLASS_BGR       = {"car":(0,230,118),"truck":(71,71,235),"bus":(255,165,0),"van":(0,215,255)}
PIX_PER_METER   = 20.0

st.markdown("""
<style>
@import url('https://fonts.googleapis.com/css2?family=Orbitron:wght@400;700;900&family=Inter:wght@300;400;600&display=swap');
.stApp{background:linear-gradient(135deg,#060610,#0d1117,#060610);color:#e2e8f0;font-family:'Inter',sans-serif;}
#MainMenu,footer,header{visibility:hidden;}
[data-testid="stSidebar"]{background:linear-gradient(180deg,#0d1117,#111827);border-right:1px solid #1e3a5f;}
.hero{background:linear-gradient(135deg,#0f172a,#1e1b4b,#0f172a);border:1px solid #312e81;border-radius:16px;padding:24px 32px;margin-bottom:20px;position:relative;overflow:hidden;}
.hero::before{content:"";position:absolute;top:0;left:0;right:0;height:3px;background:linear-gradient(90deg,#6366f1,#06b6d4,#10b981,#6366f1);background-size:200% auto;animation:shimmer 3s linear infinite;}
@keyframes shimmer{to{background-position:200% center;}}
.hero-title{font-family:'Orbitron',monospace;font-size:22px;font-weight:900;background:linear-gradient(90deg,#6366f1,#06b6d4,#10b981);-webkit-background-clip:text;-webkit-text-fill-color:transparent;margin:0 0 6px;}
.hero-sub{color:#64748b;font-size:11px;letter-spacing:2px;text-transform:uppercase;margin:0;}
.mcard{background:linear-gradient(135deg,#0f172a,#1e1b4b);border:1px solid #1e3a5f;border-radius:12px;padding:14px;text-align:center;position:relative;overflow:hidden;}
.mcard::after{content:"";position:absolute;bottom:0;left:0;right:0;height:2px;background:linear-gradient(90deg,transparent,#6366f1,transparent);}
.mval{font-family:'Orbitron',monospace;font-size:22px;font-weight:700;margin:6px 0 3px;line-height:1;}
.mlbl{font-size:9px;text-transform:uppercase;letter-spacing:2px;color:#64748b;margin:0;}
.sig-green{width:60px;height:60px;border-radius:50%;background:radial-gradient(circle at 35% 35%,#4ade80,#16a34a);box-shadow:0 0 30px #16a34a;margin:8px auto;animation:pg 2s ease-in-out infinite;}
.sig-emg{width:60px;height:60px;border-radius:50%;background:radial-gradient(circle at 35% 35%,#ff6b6b,#cc0000);box-shadow:0 0 40px #ff0000;margin:8px auto;animation:pe .5s ease-in-out infinite;}
@keyframes pg{0%,100%{box-shadow:0 0 30px #16a34a,0 0 60px rgba(22,163,74,.4);}50%{box-shadow:0 0 50px #16a34a,0 0 100px rgba(22,163,74,.6);}}
@keyframes pe{0%,100%{box-shadow:0 0 40px #ff0000;}50%{box-shadow:0 0 80px #ff0000;}}
.timer{font-family:'Orbitron',monospace;font-size:38px;font-weight:900;color:white;margin:4px 0;}
.sh{display:flex;align-items:center;margin:16px 0 10px;padding-bottom:8px;border-bottom:1px solid #1e3a5f;}
.stitle{font-family:'Orbitron',monospace;font-size:11px;font-weight:700;color:#94a3b8;letter-spacing:2px;text-transform:uppercase;margin:0;}
.emg-alert{background:linear-gradient(135deg,#1a0000,#2d0000);border:2px solid #ff0000;border-radius:12px;padding:14px;text-align:center;animation:ba 1s ease-in-out infinite;}
@keyframes ba{0%,100%{border-color:#ff0000;box-shadow:0 0 20px rgba(255,0,0,.3);}50%{border-color:#ff6666;box-shadow:0 0 40px rgba(255,0,0,.6);}}
.warn-alert{background:linear-gradient(135deg,#1a1500,#2d2200);border:1px solid #eab308;border-radius:10px;padding:10px;margin:6px 0;}
.inc-alert{background:linear-gradient(135deg,#1a0a00,#2d1500);border:1px solid #f97316;border-radius:10px;padding:10px;margin:6px 0;}
.pq-box{background:linear-gradient(135deg,#0a1a0a,#0f2d0f);border:2px solid #16a34a;border-radius:12px;padding:14px;text-align:center;box-shadow:0 0 20px rgba(22,163,74,.2);}
.fcard{background:linear-gradient(135deg,#0f172a,#1e1b4b);border:1px solid #1e3a5f;border-radius:10px;padding:14px;margin:6px 0;}
.ftitle{font-family:'Orbitron',monospace;font-size:10px;color:#6366f1;letter-spacing:2px;margin:0 0 6px;text-transform:uppercase;}
.footer{background:linear-gradient(135deg,#0f172a,#1e1b4b);border:1px solid #1e3a5f;border-radius:10px;padding:14px 20px;margin-top:24px;text-align:center;}
.footer p{color:#475569;font-size:11px;margin:3px 0;}
.footer span{color:#6366f1;font-weight:600;}
div[data-testid="stFileUploadDropzone"]{background:rgba(99,102,241,.05)!important;border:2px dashed #312e81!important;border-radius:12px!important;}
.stTabs [data-baseweb="tab"]{color:#64748b;font-family:'Orbitron',monospace;font-size:10px;letter-spacing:1px;}
.stTabs [aria-selected="true"]{color:#6366f1!important;border-bottom:2px solid #6366f1!important;}
</style>
""", unsafe_allow_html=True)

# ── Session State
_SS_DEFAULTS = [("history",[]),("lane_wait",{l:0 for l in LANE_NAMES}),
                ("trad_waits",[]),("ai_waits",[]),("speed_hist",[])]
for k,v in _SS_DEFAULTS:
    st.session_state.setdefault(k, v)

# ── Core Functions
def detect(image_bytes):
    try:
        r = requests.post(f"{RF_URL}/{MODEL_ID}",
            params={"api_key":API_KEY},
            files={"file":("img.jpg",image_bytes,"image/jpeg")},
            timeout=15)
        r.raise_for_status()
        return r.json().get("predictions",[])
    except: return []

def weighted_density(counts):
    return sum(cnt * VEHICLE_WEIGHTS.get(cls, 1.0) for cls, cnt in counts.items())

def smart_green(total, counts, wait=0, wm=1.0, tod=1.0):
    wd = weighted_density(counts)
    return min(60, max(10, round((10 + wd * 2 + wait * 0.5) * wm * tod)))

_tod_cache = {}
def tod_factor():
    h = time.localtime().tm_hour
    if h in _tod_cache: return _tod_cache[h]
    if 7<=h<=9:   r=(1.4,'Morning Peak')
    elif 12<=h<=14: r=(1.2,'Lunch Hour')
    elif 17<=h<=20: r=(1.5,'Evening Peak')
    elif h>=22 or h<6: r=(0.7,'Night')
    else: r=(1.0,'Normal')
    _tod_cache[h]=r; return r

def auto_weather():
    h = time.localtime().tm_hour
    if h>=20 or h<6: return "Night",1.2
    if random.random()<0.1: return "Rain",1.3
    return "Clear",1.0

def density_label(n):
    if n==0:   return "EMPTY",  "#94a3b8"
    if n<=4:   return "LOW",    "#4ade80"
    if n<=10:  return "MEDIUM", "#fbbf24"
    if n<=18:  return "HIGH",   "#fb923c"
    return "CRITICAL","#f87171"

def is_emg(preds):
    return any(p["class"]=="truck" and p["confidence"]>0.88 for p in preds)

def incident_check(preds, total):
    if total==0: return True,"BLOCKED LANE"
    if sum(1 for p in preds if p["class"]=="truck")>=2: return True,"INCIDENT - Multiple heavy vehicles"
    return False,""

def priority_score(total, counts, wait, emg, _wd=None):
    if emg: return 9999
    wd_val = _wd if _wd is not None else weighted_density(counts)
    return (wd_val * 2) + (wait * 1.5)

def ai_vs_trad(ai_g, total):
    tw=max(0,90-30+total*1.5); aw=max(0,90-ai_g+total*0.8)
    return tw,aw,round((tw-aw)/max(tw,1)*100,1)

def ped_safety(total):
    h=time.localtime().tm_hour
    school=(7<=h<=9)or(13<=h<=15)
    if total<=2: return 15,5,school,"LOW traffic - extended pedestrian window"
    return 0,3,school,"Normal pedestrian crossing"

# ── Centroid Tracker
class CentroidTracker:
    def __init__(self,max_dist=70,max_lost=25):
        self.next_id=0; self.objects={}; self.lost={}
        self.all_ids=set(); self.max_dist=max_dist; self.max_lost=max_lost

    def update(self,detections):
        if not detections:
            for tid in list(self.lost):
                self.lost[tid]+=1
                if self.lost[tid]>self.max_lost:
                    self.objects.pop(tid,None); self.lost.pop(tid,None)
            return dict(self.objects)
        if not self.objects:
            for cx,cy,cls in detections:
                self.objects[self.next_id]=(cx,cy,cls)
                self.lost[self.next_id]=0
                self.all_ids.add(self.next_id); self.next_id+=1
            return dict(self.objects)
        used_t=set(); used_d=set(); matched={}
        for i,(cx,cy,cls) in enumerate(detections):
            best_t=None; best_d=self.max_dist
            for tid,(ox,oy,_) in self.objects.items():
                if tid in used_t: continue
                dx2=(cx-ox)**2; dy2=(cy-oy)**2
                if dx2+dy2 >= best_d*best_d: continue
                d=math.sqrt(dx2+dy2)
                if d<best_d: best_d=d; best_t=tid
            if best_t is not None:
                matched[best_t]=(cx,cy,cls); self.objects[best_t]=(cx,cy,cls)
                self.lost[best_t]=0; used_t.add(best_t); used_d.add(i)
        for i,(cx,cy,cls) in enumerate(detections):
            if i not in used_d:
                matched[self.next_id]=(cx,cy,cls); self.objects[self.next_id]=(cx,cy,cls)
                self.lost[self.next_id]=0; self.all_ids.add(self.next_id); self.next_id+=1
        for tid in list(self.objects):
            if tid not in matched:
                self.lost[tid]=self.lost.get(tid,0)+1
                if self.lost[tid]>self.max_lost:
                    self.objects.pop(tid,None); self.lost.pop(tid,None)
                else: matched[tid]=self.objects.get(tid,(0,0,"car"))
        return matched

# ── Speed Estimator
class SpeedEstimator:
    def __init__(self,fps=25,ppm=PIX_PER_METER):
        self.fps=fps; self.ppm=ppm
        self.prev={}; self.speeds={}; self.frame_no=0

    def update(self,tracked):
        self.frame_no+=1
        for tid,(cx,cy,cls) in tracked.items():
            if tid in self.prev:
                px,py,pf=self.prev[tid]
                dt=self.frame_no-pf
                if dt>0:
                    spd=math.hypot(cx-px,cy-py)*self.fps*3.6/(self.ppm*dt)
                    self.speeds[tid]=0.7*self.speeds.get(tid,spd)+0.3*spd
            self.prev[tid]=(cx,cy,self.frame_no)
        return self.speeds

    def avg_speed(self): return round(sum(self.speeds.values())/len(self.speeds),1) if self.speeds else 0
    def max_speed(self): return round(max(self.speeds.values()),1) if self.speeds else 0
    def speeding(self,limit): return {t:s for t,s in self.speeds.items() if s>limit}

# ── Traffic Predictor
class TrafficPredictor:
    def __init__(self,window=10):
        self.history=deque(maxlen=window)

    def add(self,count): self.history.append(count)

    def predict_next(self,steps=3):
        if len(self.history)<3:
            avg=int(sum(self.history)/max(len(self.history),1))
            return [avg]*steps
        data=list(self.history); n=len(data)
        xm=(n-1)/2.0; ym=sum(data)/n
        num=sum((i-xm)*(data[i]-ym) for i in range(n))
        den=sum((i-xm)**2 for i in range(n)) or 1
        slope=num/den; intercept=ym-slope*xm
        return [max(0,min(int(intercept+slope*(n+i)),50)) for i in range(1,steps+1)]

    def congestion_risk(self):
        if len(self.history)<2: return "Unknown","#94a3b8"
        p=self.predict_next(3); avg=sum(p)/len(p)
        if avg>18: return "HIGH RISK","#f87171"
        if avg>10: return "MEDIUM RISK","#fbbf24"
        if avg>4:  return "LOW RISK","#4ade80"
        return "CLEAR","#94a3b8"

# ── Wrong Way Detector
class WrongWayDetector:
    def __init__(self,expected="down"):
        self.expected=expected; self.prev={}; self.wrong=set()

    def update(self,tracked):
        self.wrong=set()
        for tid,(cx,cy,cls) in tracked.items():
            if tid in self.prev:
                px,py=self.prev[tid]; dy=cy-py; dx=cx-px
                if abs(dy)>5 or abs(dx)>5:
                    if self.expected=="down" and dy<-15: self.wrong.add(tid)
                    elif self.expected=="up"  and dy>15: self.wrong.add(tid)
            self.prev[tid]=(cx,cy)
        return self.wrong

# ── 4-Lane Simulation from 1 image
def simulate_4lanes(real_total, real_counts, real_emg, base_wait):
    lanes=[]
    tf,_=tod_factor(); _,wm=auto_weather()
    for i,ln in enumerate(LANE_NAMES):
        if i==0:
            tot=real_total; cc=real_counts; emg=real_emg; ws=base_wait
        else:
            scale=random.uniform(0.2,0.9)
            tot=max(0,int(real_total*scale+random.randint(-2,2)))
            cc={k:max(0,int(v*scale)) for k,v in real_counts.items()}
            emg=False; ws=st.session_state.lane_wait.get(ln,0)
        wd=weighted_density(cc)
        gt=smart_green(tot,cc,ws,wm,tf)
        if emg: gt=60
        dl,dc=density_label(tot)
        if emg: dl="EMERGENCY"; dc="#ff4444"
        sc=priority_score(tot,cc,ws,emg,_wd=wd)
        lanes.append({"lane":ln,"total":tot,"counts":cc,"wd":wd,
                      "gt":gt,"density":dl,"dens_col":dc,
                      "emg":emg,"wait":ws,"score":sc})
    lanes.sort(key=lambda x:x["score"],reverse=True)
    return lanes

# ── PDF Generator
def make_pdf(history, speed_hist, metrics):
    if not PDF_OK: return None
    buf=io.BytesIO()
    doc=SimpleDocTemplate(buf,pagesize=A4,
        rightMargin=40,leftMargin=40,topMargin=40,bottomMargin=40)
    styles=getSampleStyleSheet()
    ts=ParagraphStyle("t",parent=styles["Heading1"],fontSize=16,
        textColor=colors.HexColor("#4ade80"),spaceAfter=6,fontName="Helvetica-Bold")
    ss=ParagraphStyle("s",parent=styles["Normal"],fontSize=10,
        textColor=colors.HexColor("#94a3b8"),spaceAfter=10)
    h2=ParagraphStyle("h2",parent=styles["Heading2"],fontSize=12,
        textColor=colors.HexColor("#6366f1"),spaceBefore=10,spaceAfter=6,fontName="Helvetica-Bold")
    bs=ParagraphStyle("b",parent=styles["Normal"],fontSize=9,
        textColor=colors.HexColor("#334155"),spaceAfter=4)
    tbl_style=TableStyle([
        ("BACKGROUND",(0,0),(-1,0),colors.HexColor("#1e1b4b")),
        ("TEXTCOLOR",(0,0),(-1,0),colors.white),
        ("FONTNAME",(0,0),(-1,0),"Helvetica-Bold"),
        ("FONTSIZE",(0,0),(-1,-1),9),
        ("ROWBACKGROUNDS",(0,1),(-1,-1),[colors.HexColor("#f8fafc"),colors.HexColor("#e2e8f0")]),
        ("GRID",(0,0),(-1,-1),0.5,colors.HexColor("#cbd5e1")),
        ("PADDING",(0,0),(-1,-1),5),
    ])
    story=[]
    story.append(Paragraph("AI TRAFFIC SIGNAL CONTROLLER",ts))
    story.append(Paragraph(f"Automated Report | Generated: {time.strftime('%d %B %Y %H:%M')}",ss))
    story.append(Spacer(1,10))
    if metrics:
        story.append(Paragraph("Detection Metrics",h2))
        mdata=[["Metric","Value"]]+[[k,str(v)] for k,v in metrics.items()]
        mt=Table(mdata,colWidths=[2.5*inch,4*inch])
        mt.setStyle(tbl_style)
        story.append(mt); story.append(Spacer(1,10))
    if speed_hist:
        story.append(Paragraph("Speed Analysis",h2))
        avg_s=round(sum(speed_hist)/len(speed_hist),1)
        story.append(Paragraph(
            f"Avg Speed: {avg_s} km/h | Peak: {round(max(speed_hist),1)} km/h | "
            f"Overspeed events: {sum(1 for s in speed_hist if s>60)}",bs))
        story.append(Spacer(1,8))
    if history:
        story.append(Paragraph("Detection Log (Last 20)",h2))
        cols=["Time","Vehicles","Weighted","Density","Green(s)","Saved(s)","Emergency"]
        ldata=[cols]+[[str(h.get(c,"--")) for c in cols] for h in history[-20:]]
        lt=Table(ldata,colWidths=[0.7*inch]*7)
        lt.setStyle(tbl_style)
        story.append(lt); story.append(Spacer(1,10))
    story.append(Paragraph("_"*70,bs))
    story.append(Paragraph(
        "Sugandha Srivastava | Sudhanshu Shukla | Sakshi Chaturvedi | Shreya Dutta",bs))
    story.append(Paragraph(
        "Dept. of CS&E | Nitra Technical Campus | Guide: Dr. Rajesh Kumar",bs))
    doc.build(story)
    buf.seek(0)
    return buf.read()

# ── HELPER: build lane card HTML cleanly (FIX for </div> bug)
def lane_card_html(ld, is_active, rank):
    sig_color  = "#4ade80" if is_active else "#f87171"
    sig_label  = "GREEN"   if is_active else "RED"
    if ld["emg"] and is_active:
        sig_color="#ff4444"; sig_label="EMERGENCY"
    gt_display = str(ld["gt"])+"s" if is_active else "WAIT"
    _DENS_COLORS={"LOW":"#4ade80","MEDIUM":"#fbbf24","HIGH":"#fb923c",
                  "CRITICAL":"#f87171","EMERGENCY":"#ff4444","EMPTY":"#94a3b8"}
    dens_col   = _DENS_COLORS.get(ld["density"],"#94a3b8")
    bg_grad  = "linear-gradient(135deg,#0a1a0a,#0f2d0f)" if is_active else "linear-gradient(135deg,#1a0a0a,#2d0f0f)"
    border   = "2px solid #16a34a" if is_active else "1px solid #7f1d1d"

    # Build HTML using concatenation — avoids </div> rendering bug
    html  = f'<div style="background:{bg_grad};border:{border};border-radius:12px;padding:14px;text-align:center;">'
    html += f'<p style="font-family:Orbitron,monospace;color:white;font-size:13px;font-weight:700;margin:0;">{LANE_ARROW[ld["lane"]]} {ld["lane"]}</p>'
    html += f'<p style="font-family:Orbitron,monospace;color:{sig_color};font-size:12px;font-weight:700;margin:4px 0;">{sig_label}</p>'
    html += f'<p style="font-family:Orbitron,monospace;color:white;font-size:22px;font-weight:900;margin:2px 0;">{gt_display}</p>'
    html += f'<hr style="border-color:#1e3a5f;margin:6px 0;">'
    html += f'<p style="color:{dens_col};font-size:11px;font-weight:700;margin:2px 0;">{ld["density"]}</p>'
    html += f'<p style="color:#64748b;font-size:10px;margin:1px 0;">{ld["total"]} vehicles | {ld["wd"]:.1f}w</p>'
    html += f'<p style="color:#f97316;font-size:10px;font-weight:700;margin:1px 0;">Score: {ld["score"]:.0f}</p>'
    html += f'<p style="color:#475569;font-size:9px;margin:1px 0;">Wait: {ld["wait"]}s | Rank: #{rank}</p>'
    if ld["emg"]:
        html += '<p style="color:#ff4444;font-size:9px;font-weight:700;margin:2px 0;">EMERGENCY VEHICLE</p>'
    html += '</div>'
    return html

# ── HERO
st.markdown("""
<div class="hero">
<p class="hero-title">TRUE AI TRAFFIC SIGNAL CONTROLLER</p>
<p class="hero-sub">Speed | Prediction | Wrong-Way | Pedestrian | Multi-Junction | PDF | 19 Features</p>
<span style="display:inline-block;background:rgba(99,102,241,.15);border:1px solid #6366f1;color:#a5b4fc;padding:3px 14px;border-radius:20px;font-size:11px;margin-top:8px;letter-spacing:1px;">UPLOAD IMAGE OR VIDEO - EVERYTHING AUTO</span>
</div>
""", unsafe_allow_html=True)

# ── SIDEBAR
tf_s,tod_s=tod_factor(); wn_s,wm_s=auto_weather(); h_now=time.localtime().tm_hour
with st.sidebar:
    st.markdown("<p style='font-family:Orbitron,monospace;font-size:11px;color:#6366f1;letter-spacing:2px;text-align:center;padding:8px 0;'>SYSTEM PANEL</p>",unsafe_allow_html=True)
    st.divider()
    st.markdown(f"""<div style="background:rgba(99,102,241,.08);border:1px solid #312e81;border-radius:10px;padding:12px;">
    <p style="color:#a5b4fc;font-size:10px;font-family:Orbitron,monospace;letter-spacing:2px;margin:0 0 8px;">AUTO-DETECTED</p>
    <p style="color:#94a3b8;font-size:11px;margin:3px 0;">Time: {h_now:02d}:00 - <span style="color:#fbbf24;">{tod_s}</span></p>
    <p style="color:#94a3b8;font-size:11px;margin:3px 0;">Weather: <span style="color:#a78bfa;">{wn_s}</span></p>
    <p style="color:#94a3b8;font-size:11px;margin:3px 0;">TOD: <span style="color:#60a5fa;">{tf_s}x</span> | Weather: <span style="color:#4ade80;">{wm_s}x</span></p>
    </div>""",unsafe_allow_html=True)
    st.divider()
    st.markdown("<p style='font-family:Orbitron,monospace;font-size:10px;color:#6366f1;letter-spacing:2px;'>VEHICLE WEIGHTS</p>",unsafe_allow_html=True)
    st.markdown("""<div style="background:rgba(0,0,0,.3);border:1px solid #1e3a5f;border-radius:8px;padding:10px;">
    <p style="color:#f87171;font-size:11px;margin:2px 0;">Truck = 3.0 units</p>
    <p style="color:#60a5fa;font-size:11px;margin:2px 0;">Bus = 2.5 units</p>
    <p style="color:#fbbf24;font-size:11px;margin:2px 0;">Van = 1.5 units</p>
    <p style="color:#4ade80;font-size:11px;margin:2px 0;">Car = 1.0 units</p>
    </div>""",unsafe_allow_html=True)
    st.divider()
    st.markdown("<p style='font-family:Orbitron,monospace;font-size:10px;color:#6366f1;letter-spacing:2px;'>AI FORMULA</p>",unsafe_allow_html=True)
    st.code("Green = 10\n+ (weighted x 2)\n+ (wait x 0.5)\nx weather x tod",language="python")
    st.divider()
    speed_limit=st.slider("Speed Alert (km/h)",20,120,60,5)
    st.divider()
    st.markdown("<div style='text-align:center;'><p style='color:#475569;font-size:10px;'>Nitra Technical Campus<br><span style='color:#6366f1;'>Guide: Dr. Rajesh Kumar</span></p></div>",unsafe_allow_html=True)

# ── TABS
tab1,tab2,tab3,tab4 = st.tabs([
    "📸 Smart Detection",
    "🎬 Video Analysis",
    "🗺️ Multi-Intersection",
    "📊 Analytics & PDF"
])

# ════════════════════════════════════
#  TAB 1 — IMAGE DETECTION
# ════════════════════════════════════
with tab1:
    st.markdown("<div class='sh'><p class='stitle'>Upload Traffic Image - All Features Auto</p></div>",unsafe_allow_html=True)
    uploaded=st.file_uploader("Drop image",type=["jpg","jpeg","png"],label_visibility="collapsed")

    if uploaded:
        img_bytes=uploaded.read()
        with st.spinner("AI analysing..."):
            preds=detect(img_bytes)
            total=len(preds)
            counts=dict(Counter(p["class"] for p in preds))
            wd=weighted_density(counts)
            tf,tod_lbl=tod_factor()
            wname,wm=auto_weather()
            wait_s=st.session_state.lane_wait.get("NORTH",0)
            gt=smart_green(total,counts,wait_s,wm,tf)
            trad_w,ai_w,imp=ai_vs_trad(gt,total)
            dl,dc=density_label(total)
            emg=is_emg(preds)
            inc,inc_msg=incident_check(preds,total)
            if emg: gt=60; dl="EMERGENCY"
            if total==0: gt=30
            score=priority_score(total,counts,wait_s,emg,_wd=wd)
            ped_g,yel_t,school,ped_note=ped_safety(total)
            if "img_pred" not in st.session_state:
                st.session_state.img_pred=TrafficPredictor()
            st.session_state.img_pred.add(total)
            pred_next=st.session_state.img_pred.predict_next(3)
            cong_risk,cr_col=st.session_state.img_pred.congestion_risk()
            lanes_4=simulate_4lanes(total,counts,emg,wait_s)
            active_lane=lanes_4[0]
            rank_map={ld["lane"]:i+1 for i,ld in enumerate(lanes_4)}

        st.session_state.trad_waits.append(trad_w)
        st.session_state.ai_waits.append(ai_w)
        st.session_state.history.append({
            "Time":time.strftime("%H:%M:%S"),"Vehicles":total,"Weighted":round(wd,1),
            "Density":dl,"Green(s)":gt,"Trad(s)":30,"Saved(s)":round(trad_w-ai_w,1),
            "Improvement":f"{imp}%","Weather":wname,"Period":tod_lbl,
            "Emergency":"YES" if emg else "No","Score":round(score,1),
        })

        # Alerts
        if total==0:
            st.markdown("<div class='warn-alert'><p style='color:#eab308;font-size:13px;font-weight:700;margin:0;'>FAIL-SAFE: No detection. Using fixed 30s timing.</p></div>",unsafe_allow_html=True)
        if emg:
            st.markdown("""<div class="emg-alert">
            <p style="font-family:Orbitron,monospace;color:#ff4444;font-size:15px;font-weight:900;margin:0;">EMERGENCY VEHICLE - INSTANT GREEN OVERRIDE</p>
            <p style="color:#ff8888;font-size:12px;margin:6px 0 0;">60s priority | All other lanes RED | Override active</p>
            </div>""",unsafe_allow_html=True)
            st.markdown("<br>",unsafe_allow_html=True)
        if inc and not emg:
            st.markdown(f"<div class='inc-alert'><p style='color:#f97316;font-size:12px;font-weight:600;margin:0;'>INCIDENT: {inc_msg}</p></div>",unsafe_allow_html=True)
        if school:
            st.markdown(f"<div class='warn-alert'><p style='color:#eab308;font-size:12px;margin:0;'>SCHOOL HOURS: Yellow extended to {yel_t}s | Pedestrian safety ON</p></div>",unsafe_allow_html=True)

        # Metrics
        st.markdown("<div class='sh'><p class='stitle'>AI Computed Metrics</p></div>",unsafe_allow_html=True)
        m1,m2,m3,m4,m5,m6,m7,m8=st.columns(8)
        dc_="#f87171" if dl in ["HIGH","CRITICAL","EMERGENCY"] else "#fbbf24" if dl=="MEDIUM" else "#4ade80"
        for col,lbl,val,ch in [
            (m1,"Vehicles",str(total),"#4ade80"),
            (m2,"Weighted",f"{wd:.1f}","#a78bfa"),
            (m3,"Density",dl,dc_),
            (m4,"AI Green",f"{gt}s","#60a5fa"),
            (m5,"Trad Green","30s","#94a3b8"),
            (m6,"Saved",f"{round(trad_w-ai_w,1)}s","#4ade80"),
            (m7,"Improvement",f"{imp}%","#fbbf24"),
            (m8,"Score",f"{score:.0f}","#f97316"),
        ]:
            with col:
                st.markdown(f'<div class="mcard"><div class="mval" style="color:{ch};font-size:16px;">{val}</div><div class="mlbl">{lbl}</div></div>',unsafe_allow_html=True)

        st.markdown("<br>",unsafe_allow_html=True)

        # Image + Signal
        ci,_,cs=st.columns([3,.05,1.1])
        with ci:
            img=np.array(Image.open(BytesIO(img_bytes)).convert("RGB"))
            fig,ax=plt.subplots(figsize=(11,7),facecolor="#0d1117")
            ax.imshow(img); ax.axis("off")
            for p in preds:
                x=p["x"]-p["width"]/2; y=p["y"]-p["height"]/2
                eg=p["class"]=="truck" and p["confidence"]>0.88
                c="#ff0000" if eg else CLASS_COLORS.get(p["class"],"#aaa")
                rect=patches.FancyBboxPatch((x,y),p["width"],p["height"],
                    linewidth=3 if eg else 2,edgecolor=c,facecolor="none",boxstyle="round,pad=2")
                ax.add_patch(rect)
                wt=VEHICLE_WEIGHTS.get(p["class"],1.0)
                ax.text(x+2,y-8,
                    f"{'EMG ' if eg else ''}{p['class'].upper()} {p['confidence']*100:.0f}% w={wt}",
                    color="black",fontsize=7.5,fontweight="bold",
                    bbox=dict(facecolor=c,edgecolor="none",pad=2,boxstyle="round,pad=0.3"))
            buf2=BytesIO()
            plt.savefig(buf2,format="png",dpi=130,bbox_inches="tight",facecolor="#0d1117")
            st.image(buf2,use_container_width=True)
            plt.close()

        with cs:
            lc="sig-emg" if emg else "sig-green"
            lt="EMERGENCY" if emg else "GREEN"
            sig_col="#ff4444" if emg else "#4ade80"
            st.markdown(
                '<div style="background:linear-gradient(135deg,#0f172a,#1a1a2e);border:1px solid #1e3a5f;border-radius:14px;padding:16px;text-align:center;">'
                '<p style="font-family:Orbitron,monospace;font-size:9px;color:#475569;letter-spacing:2px;margin:0 0 4px;">AI SIGNAL DECISION</p>'
                f'<div class="{lc}"></div>'
                f'<p style="font-family:Orbitron,monospace;font-size:15px;font-weight:700;color:{sig_col};letter-spacing:2px;margin:0;">{lt}</p>'
                f'<p class="timer">{gt}s</p>'
                '<hr style="border-color:#1e3a5f;margin:8px 0;">'
                f'<p style="color:{dc};font-size:14px;font-weight:700;margin:0;">{dl}</p>'
                f'<p style="color:#64748b;font-size:10px;margin:2px 0;">{total} vehicles | {wd:.1f} weighted</p>'
                '<hr style="border-color:#1e3a5f;margin:6px 0;">'
                '<p style="color:#475569;font-size:9px;margin:0 0 2px;">PERIOD</p>'
                f'<p style="color:#fbbf24;font-size:12px;font-weight:700;margin:0;">{tod_lbl} | {wname}</p>'
                '<hr style="border-color:#1e3a5f;margin:6px 0;">'
                '<p style="color:#475569;font-size:9px;margin:0 0 2px;">PREDICTION</p>'
                f'<p style="color:#60a5fa;font-size:11px;font-weight:700;margin:0;">{pred_next}</p>'
                f'<p style="color:{cr_col};font-size:10px;font-weight:700;margin:2px 0;">{cong_risk}</p>'
                '<hr style="border-color:#1e3a5f;margin:6px 0;">'
                f'<p style="color:#a78bfa;font-size:10px;margin:0;">Ped yellow: {yel_t}s | Score: {score:.0f}</p>'
                f'<p style="color:#4ade80;font-size:10px;margin:2px 0;">Saved: {round(trad_w-ai_w,1)}s ({imp}%)</p>'
                '</div>',
            unsafe_allow_html=True)

        # Weighted charts
        if counts:
            st.markdown("<br>",unsafe_allow_html=True)
            st.markdown("<div class='sh'><p class='stitle'>Weighted Vehicle Analysis</p></div>",unsafe_allow_html=True)
            ca,cb=st.columns([1,2])
            with ca:
                for cls,cnt in counts.items():
                    ch=CLASS_COLORS.get(cls,"#fff"); wt=VEHICLE_WEIGHTS.get(cls,1.0)
                    ww=cnt*wt; pct=int(ww/max(wd,1)*100)
                    st.markdown(
                        f'<div style="display:flex;justify-content:space-between;align-items:center;padding:8px 12px;background:rgba(255,255,255,.03);border-radius:8px;margin:4px 0;border-left:3px solid {ch};">'
                        f'<span style="color:{ch};font-weight:600;font-size:12px;">{cls.upper()}</span>'
                        f'<span style="color:#64748b;font-size:11px;">{cnt} x {wt} =</span>'
                        f'<span style="color:white;font-family:Orbitron,monospace;font-size:14px;font-weight:700;">{ww:.1f}</span>'
                        f'</div>'
                        f'<div style="background:#1e293b;border-radius:4px;height:5px;margin:-2px 0 6px;">'
                        f'<div style="background:{ch};width:{pct}%;height:5px;border-radius:4px;"></div>'
                        f'</div>',
                    unsafe_allow_html=True)
                st.markdown(f'<p style="color:#6366f1;font-size:12px;text-align:center;margin-top:8px;">Total weighted: <b>{wd:.1f}</b></p>',unsafe_allow_html=True)
            with cb:
                cl=list(counts.keys()); cv=list(counts.values())
                wv=[counts.get(c,0)*VEHICLE_WEIGHTS.get(c,1.0) for c in cl]
                clrs=[CLASS_COLORS.get(c,"#888") for c in cl]
                fig3,(ax3a,ax3b)=plt.subplots(1,2,figsize=(9,3.5),facecolor="#0d1117")
                for axx,data,ttl in [(ax3a,cv,"Raw Count"),(ax3b,wv,"Weighted Units")]:
                    axx.set_facecolor("#111827")
                    bars=axx.bar(cl,data,color=clrs,edgecolor="#1e293b",linewidth=1.5,width=0.5)
                    for bar,val2 in zip(bars,data):
                        axx.text(bar.get_x()+bar.get_width()/2,bar.get_height()+.05,
                            f"{val2:.1f}" if "Weighted" in ttl else str(int(val2)),
                            ha="center",va="bottom",color="white",fontsize=11,fontweight="bold")
                    axx.set_ylim(0,max(data)*1.4 if data else 1)
                    axx.tick_params(colors="#475569")
                    axx.set_xticklabels([c.upper() for c in cl],color="#94a3b8")
                    for sp in axx.spines.values(): sp.set_color("#1e3a5f")
                    axx.yaxis.set_visible(False)
                    axx.set_title(ttl,color="#64748b",fontsize=10,pad=8)
                plt.tight_layout()
                b3=BytesIO()
                plt.savefig(b3,format="png",dpi=120,bbox_inches="tight",facecolor="#0d1117")
                st.image(b3,use_container_width=True)
                plt.close()

        # Prediction cards
        st.markdown("<br>",unsafe_allow_html=True)
        st.markdown("<div class='sh'><p class='stitle'>Traffic Prediction - Next 3 Steps</p></div>",unsafe_allow_html=True)
        pc1,pc2,pc3,pc4,pc5=st.columns(5)
        for col,val,lbl in zip([pc1,pc2,pc3],pred_next,["Next 1","Next 2","Next 3"]):
            ch="#f87171" if val>18 else "#fbbf24" if val>10 else "#4ade80"
            with col:
                st.markdown(f'<div class="mcard"><div class="mval" style="color:{ch};">{val}</div><div class="mlbl">{lbl} vehicles</div></div>',unsafe_allow_html=True)
        with pc4:
            st.markdown(f'<div class="mcard"><div class="mval" style="color:{cr_col};font-size:14px;">{cong_risk}</div><div class="mlbl">CONGESTION RISK</div></div>',unsafe_allow_html=True)
        with pc5:
            rec="Extend green" if cong_risk=="HIGH RISK" else "Reduce green" if cong_risk=="CLEAR" else "Maintain"
            st.markdown(f'<div class="mcard"><div class="mval" style="color:#6366f1;font-size:12px;">{rec}</div><div class="mlbl">RECOMMENDATION</div></div>',unsafe_allow_html=True)

        # 4-Lane Auto Decision
        st.markdown("<br>",unsafe_allow_html=True)
        st.markdown("<div class='sh'><p class='stitle'>Auto 4-Lane Intersection Decision</p></div>",unsafe_allow_html=True)
        reason="EMERGENCY OVERRIDE" if active_lane["emg"] else f"Highest priority score: {active_lane['score']:.1f}"

        st.markdown(
            '<div class="pq-box">'
            '<p style="color:#475569;font-size:10px;letter-spacing:2px;margin:0 0 6px;">AI AUTO PRIORITY DECISION</p>'
            f'<p style="font-family:Orbitron,monospace;color:#4ade80;font-size:20px;font-weight:900;margin:0;">{active_lane["lane"]} LANE - GREEN ({active_lane["gt"]}s)</p>'
            f'<p style="color:#64748b;font-size:12px;margin:8px 0 0;">{reason} | Weather: {wname} | Period: {tod_lbl} | All other lanes: RED</p>'
            '</div>',
        unsafe_allow_html=True)

        queue_parts=[f"{'GREEN' if i==0 else 'RED'} {ld['lane']}(score={ld['score']:.0f})" for i,ld in enumerate(lanes_4)]
        st.markdown(f'<p style="color:#475569;font-size:11px;text-align:center;margin:8px 0;">QUEUE: {" > ".join(queue_parts)}</p>',unsafe_allow_html=True)
        st.markdown("<br>",unsafe_allow_html=True)

        # Lane cards — using helper function (FIX)
        lc1,lc2,lc3,lc4=st.columns(4)
        for col,ld in zip([lc1,lc2,lc3,lc4],lanes_4):
            is_act=(ld["lane"]==active_lane["lane"])
            rank=rank_map[ld["lane"]]
            with col:
                st.markdown(lane_card_html(ld,is_act,rank),unsafe_allow_html=True)

        # Update waiting times
        for ld in lanes_4:
            if ld["lane"]!=active_lane["lane"]:
                st.session_state.lane_wait[ld["lane"]]=st.session_state.lane_wait.get(ld["lane"],0)+active_lane["gt"]
            else:
                st.session_state.lane_wait[ld["lane"]]=0

    else:
        st.markdown("""<div style="background:linear-gradient(135deg,#0f172a,#1e1b4b);border:2px dashed #312e81;border-radius:16px;padding:56px;text-align:center;">
        <p style="font-size:56px;margin:0;">📸</p>
        <p style="font-family:Orbitron,monospace;color:#6366f1;font-size:18px;margin:14px 0 6px;">UPLOAD ANY TRAFFIC IMAGE</p>
        <p style="color:#475569;font-size:13px;margin:0;">Auto detects | Weighted scores | Lane priority | Emergency | Pedestrian | Prediction | 4-lane auto</p>
        </div>""",unsafe_allow_html=True)

# ════════════════════════════════════
#  TAB 2 — VIDEO ANALYSIS
# ════════════════════════════════════
with tab2:
    st.markdown("<div class='sh'><p class='stitle'>Upload Traffic Video - Speed + Wrong-Way + All Features</p></div>",unsafe_allow_html=True)
    vc1,vc2,vc3,vc4=st.columns(4)
    with vc1: sim_mode=st.selectbox("Scenario",["Normal Traffic","Peak Hour","Emergency Mode","Night Mode"])
    with vc2: every_n=st.select_slider("Frame skip",options=[3,5,8,10,15],value=5)
    with vc3: conf_v=st.slider("Confidence",0.1,0.9,0.35,0.05)
    with vc4: loop_v=st.checkbox("CCTV Loop",value=False)

    MODE_F={"Normal Traffic":{"wm":1.0,"tod":1.0,"lbl":"Normal","col":"#4ade80"},
            "Peak Hour":{"wm":1.3,"tod":1.5,"lbl":"Peak","col":"#f97316"},
            "Emergency Mode":{"wm":1.0,"tod":1.0,"lbl":"Emergency","col":"#ff4444"},
            "Night Mode":{"wm":1.2,"tod":0.7,"lbl":"Night","col":"#a78bfa"}}
    mf=MODE_F[sim_mode]; force_emg=(sim_mode=="Emergency Mode")

    st.markdown(
        f'<div style="background:rgba(99,102,241,.08);border:1px solid #312e81;border-radius:10px;padding:10px 16px;margin:6px 0;">'
        f'<p style="color:{mf["col"]};font-family:Orbitron,monospace;font-size:12px;font-weight:700;margin:0;">'
        f'MODE: {mf["lbl"].upper()} | Weather: {mf["wm"]}x | TOD: {mf["tod"]}x | Skip: {every_n} frames | Speed alert: {speed_limit} km/h'
        f'</p></div>',
    unsafe_allow_html=True)

    uploaded_vid=st.file_uploader("Drop video",type=["mp4","avi","mov"],label_visibility="collapsed",key="vid_up")

    if uploaded_vid:
        with tempfile.NamedTemporaryFile(delete=False,suffix=".mp4") as tmp:
            tmp.write(uploaded_vid.read()); vid_path=tmp.name

        cap=cv2.VideoCapture(vid_path)
        fps=cap.get(cv2.CAP_PROP_FPS) or 25
        fw=int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
        fh=int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
        total_f=int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

        out_path="/content/output_video.mp4"
        writer=cv2.VideoWriter(out_path,cv2.VideoWriter_fourcc(*"mp4v"),fps,(fw+320,fh))

        tracker=CentroidTracker()
        speed_est=SpeedEstimator(fps=fps)
        wrong_way=WrongWayDetector()
        predictor=TrafficPredictor()

        cur_preds=[]; cur_total=0; cur_gt=30; cur_dens="LOW"
        cur_dc=(0,230,118); cur_counts={}; cur_wd=0.0
        cur_emg=False; cur_speeds={}; wait_v=0.0
        frame_idx=0; api_calls=0; alert_msgs=[]
        tl_v=[]; tl_g=[]; tl_w=[]; tl_s=[]

        prog=st.progress(0,"Processing..."); stat=st.empty()

        while True:
            ret,frame=cap.read()
            if not ret:
                if loop_v:
                    cap.set(cv2.CAP_PROP_POS_FRAMES,0)
                    ret,frame=cap.read()
                    if not ret: break
                else: break

            frame_idx+=1
            pad=np.zeros((fh,320,3),dtype=np.uint8); pad[:]=np.array([8,8,18])
            frame_draw=np.hstack([frame.copy(),pad])

            if frame_idx%every_n==0:
                _,enc=cv2.imencode(".jpg",frame[:fh,:fw],[cv2.IMWRITE_JPEG_QUALITY,82])
                try:
                    res=requests.post(f"{RF_URL}/{MODEL_ID}",
                        params={"api_key":API_KEY,"confidence":conf_v},
                        files={"file":("f.jpg",enc.tobytes(),"image/jpeg")},timeout=15)
                    raw=res.json().get("predictions",[]) if res.status_code==200 else cur_preds
                    dets=[(int(p["x"]),int(p["y"]),p.get("class","car")) for p in raw]
                    tracked=tracker.update(dets)
                    cur_preds=raw; cur_total=len(cur_preds)
                    cur_counts={}
                    for p in cur_preds:
                        cls=p.get("class","car")
                        cur_counts[cls]=cur_counts.get(cls,0)+1
                    cur_wd=sum(cnt*VEHICLE_WEIGHTS.get(c,1.0) for c,cnt in cur_counts.items())
                    cur_speeds=speed_est.update(tracked)
                    avg_s=speed_est.avg_speed()
                    if avg_s>0: tl_s.append(avg_s)
                    ww_ids=wrong_way.update(tracked)
                    predictor.add(cur_total)
                    if cur_total==0: wait_v+=every_n/fps
                    else: wait_v=max(0,wait_v-1)
                    cur_emg=force_emg or is_emg(cur_preds)
                    cur_gt=smart_green(cur_total,cur_counts,wait_v,mf["wm"],mf["tod"])
                    if cur_emg: cur_gt=60
                    cur_dens,dc_hex=density_label(cur_total)
                    cur_dc=tuple(int(dc_hex.lstrip("#")[i:i+2],16) for i in (4,2,0))
                    if cur_emg: cur_dens="EMERGENCY"
                    alert_msgs=[]
                    if cur_emg: alert_msgs.append("EMERGENCY VEHICLE - INSTANT GREEN")
                    if ww_ids: alert_msgs.append(f"WRONG-WAY: {len(ww_ids)} vehicle(s)")
                    spd_alerts=speed_est.speeding(speed_limit)
                    if spd_alerts: alert_msgs.append(f"OVERSPEED: {len(spd_alerts)} vehicle(s)")
                    if cur_total>18: alert_msgs.append(f"CRITICAL CONGESTION: {cur_total} vehicles")
                    tl_v.append(cur_total); tl_g.append(cur_gt); tl_w.append(cur_wd)
                    api_calls+=1
                    stat.markdown(
                        f"Frame **{frame_idx}/{total_f}** | Vehicles: **{cur_total}** | "
                        f"Speed: **{speed_est.avg_speed():.0f} km/h** | "
                        f"Wrong-way: **{len(ww_ids)}** | Green: **{cur_gt}s**"
                    )
                except: pass

            # Draw boxes
            for p in cur_preds:
                x1=int(p["x"]-p["width"]/2); y1=int(p["y"]-p["height"]/2)
                x2=int(p["x"]+p["width"]/2); y2=int(p["y"]+p["height"]/2)
                cls=p.get("class","car"); cf=p.get("confidence",0)
                eg=cls=="truck" and cf>0.88
                c=(0,0,255) if (eg or cur_emg) else CLASS_BGR.get(cls,(180,180,180))
                cv2.rectangle(frame_draw,(x1,y1),(x2,y2),c,3 if eg else 2)
                wt=VEHICLE_WEIGHTS.get(cls,1.0)
                lbl=f"{'[EMG]' if eg else ''}{cls.upper()} {cf*100:.0f}% w={wt}"
                (tw,th),_=cv2.getTextSize(lbl,cv2.FONT_HERSHEY_SIMPLEX,0.42,1)
                cv2.rectangle(frame_draw,(x1,y1-th-8),(x1+tw+6,y1),c,-1)
                cv2.putText(frame_draw,lbl,(x1+3,y1-4),cv2.FONT_HERSHEY_SIMPLEX,0.42,(0,0,0),1,cv2.LINE_AA)
                cx_=int(p["x"]); cy_=int(p["y"])
                for tid,(ox,oy,_) in tracker.objects.items():
                    if math.hypot(cx_-ox,cy_-oy)<15:
                        spd=cur_speeds.get(tid,0)
                        ww_f=" [WW]" if tid in (wrong_way.wrong if hasattr(wrong_way,"wrong") else set()) else ""
                        cv2.putText(frame_draw,f"#{tid} {spd:.0f}km/h{ww_f}",
                            (x1,y2+13),cv2.FONT_HERSHEY_SIMPLEX,0.38,c,1,cv2.LINE_AA)
                        break

            # Alert banners
            for i,msg in enumerate(alert_msgs[:2]):
                al_c=(0,0,180) if "EMERGENCY" in msg or "WRONG" in msg else (0,60,130)
                cv2.rectangle(frame_draw,(0,i*34),(fw,34+i*34),al_c,-1)
                cv2.putText(frame_draw,f"  {msg}",(8,22+i*34),cv2.FONT_HERSHEY_SIMPLEX,0.5,(255,255,255),2,cv2.LINE_AA)

            # Overlay panel
            ox=fw+8
            def ptxt(tx,x,y,sc=0.44,col=(200,200,200),th=1):
                cv2.putText(frame_draw,tx,(x,y),cv2.FONT_HERSHEY_SIMPLEX,sc,col,th,cv2.LINE_AA)

            sig_c=(0,230,118) if not cur_emg else (0,0,255)
            mc={"Normal Traffic":(0,230,118),"Peak Hour":(0,165,255),
                "Emergency Mode":(0,0,255),"Night Mode":(180,100,255)}.get(sim_mode,(200,200,200))
            ptxt(f"MODE: {mf['lbl'].upper()}",ox,22,0.48,mc,2)
            ptxt("NORTH LANE",ox,42,0.5,sig_c,2)
            cv2.circle(frame_draw,(fw+285,32),19,(20,20,40),-1)
            cv2.circle(frame_draw,(fw+285,32),17,sig_c,-1)
            ptxt(f"{'EMERGENCY' if cur_emg else 'GREEN'}  {cur_gt}s",ox,60,0.52,sig_c,2)
            cv2.line(frame_draw,(ox-3,70),(fw+317,70),(50,50,80),1)
            ptxt(f"Vehicles: {cur_total}  Weighted: {cur_wd:.1f}",ox,88,0.42,cur_dc)
            ptxt(f"Density: {cur_dens}",ox,105,0.42,cur_dc)
            ptxt(f"Wait: {int(wait_v)}s",ox,122,0.40,(150,200,180))
            cv2.line(frame_draw,(ox-3,132),(fw+317,132),(50,50,80),1)
            avg_spd=speed_est.avg_speed(); max_spd=speed_est.max_speed()
            spd_c=(0,0,200) if avg_spd>speed_limit else (200,100,255)
            ptxt(f"Avg Speed: {avg_spd:.0f} km/h",ox,150,0.42,spd_c)
            ptxt(f"Max Speed: {max_spd:.0f} km/h",ox,167,0.42,spd_c)
            ptxt(f"Overspeed: {len(speed_est.speeding(speed_limit))}",ox,184,0.42,spd_c)
            cv2.line(frame_draw,(ox-3,194),(fw+317,194),(50,50,80),1)
            ww_ids2=wrong_way.wrong if hasattr(wrong_way,"wrong") else set()
            ww_c=(0,0,200) if ww_ids2 else (0,230,118)
            ptxt(f"Wrong-Way: {len(ww_ids2)}",ox,212,0.42,ww_c)
            ptxt(f"Tracked: {len(tracker.objects)}  Unique: {len(tracker.all_ids)}",ox,228,0.38,(150,180,200))
            cv2.line(frame_draw,(ox-3,238),(fw+317,238),(50,50,80),1)
            ptxt("CLASS  CNT  WT",ox,255,0.35,(100,116,139))
            y_=270
            dc4={"car":(0,230,118),"truck":(71,71,235),"bus":(255,165,0),"van":(0,215,255)}
            for cls_,cnt_ in cur_counts.items():
                cc_=dc4.get(cls_,(180,180,180)); wt_=VEHICLE_WEIGHTS.get(cls_,1.0)
                cv2.rectangle(frame_draw,(ox,y_-10),(ox+5,y_+2),cc_,-1)
                ptxt(f"{cls_.upper()} {cnt_} x{wt_}={cnt_*wt_:.1f}",ox+10,y_,0.38,cc_)
                y_+=16
            cv2.line(frame_draw,(ox-3,y_+3),(fw+317,y_+3),(50,50,80),1)
            ptxt(f"Green=10+({cur_wd:.0f}x2)+({int(wait_v)}x0.5)={cur_gt}s",ox,y_+18,0.35,(99,102,241))
            bar_w=min(int(len(tracker.all_ids)/50*290),290)
            cv2.rectangle(frame_draw,(ox,y_+26),(fw+314,y_+33),(30,30,50),-1)
            if bar_w>0: cv2.rectangle(frame_draw,(ox,y_+26),(ox+bar_w,y_+33),sig_c,-1)
            cv2.putText(frame_draw,f"F:{frame_idx} API:{api_calls}",(10,fh-10),cv2.FONT_HERSHEY_SIMPLEX,0.38,(100,100,180),1,cv2.LINE_AA)
            if cur_emg:
                cv2.rectangle(frame_draw,(ox-3,fh-52),(fw+317,fh-3),(0,0,55),-1)
                ptxt("EMERGENCY VEHICLE",ox,fh-30,0.46,(80,80,255),2)
                ptxt("ALL LANES: RED",ox,fh-10,0.40,(150,150,255))

            writer.write(frame_draw)
            prog.progress(min(frame_idx/max(1,total_f),1.0))
            if loop_v and frame_idx>=total_f*2: break

        cap.release(); writer.release()
        prog.progress(1.0,"Done!")
        avg_v7=round(sum(tl_v)/max(len(tl_v),1),1)
        avg_g7=round(sum(tl_g)/max(len(tl_g),1),1)
        avg_s7=round(sum(tl_s)/max(len(tl_s),1),1) if tl_s else 0
        max_v7=max(tl_v) if tl_v else 0
        st.session_state.speed_hist.extend(tl_s)
        st.success(f"Done! {frame_idx} frames | {api_calls} detections | {len(tracker.all_ids)} unique vehicles | Avg speed: {avg_s7:.0f} km/h")

        r1,r2,r3,r4,r5,r6=st.columns(6)
        for col,lbl,val,ch in [
            (r1,"Frames",str(frame_idx),"#4ade80"),
            (r2,"Detections",str(api_calls),"#60a5fa"),
            (r3,"Avg Vehicles",str(avg_v7),"#fb923c"),
            (r4,"Max Vehicles",str(max_v7),"#f87171"),
            (r5,"Unique IDs",str(len(tracker.all_ids)),"#a78bfa"),
            (r6,"Avg Speed",f"{avg_s7:.0f} km/h","#c026d3"),
        ]:
            with col:
                st.markdown(f'<div class="mcard"><div class="mval" style="color:{ch};font-size:18px;">{val}</div><div class="mlbl">{lbl}</div></div>',unsafe_allow_html=True)

        if len(tl_v)>=2:
            st.markdown("<br>",unsafe_allow_html=True)
            st.markdown("<div class='sh'><p class='stitle'>Analytics Graphs</p></div>",unsafe_allow_html=True)
            n_p=4 if tl_s else 3
            fig_v,axs=plt.subplots(1,n_p,figsize=(20,4.5),facecolor="#0d1117")
            pd_list=[(tl_v,"#4ade80","Vehicle Count","Vehicles"),
                     (tl_g,"#60a5fa","Green Time","Seconds"),
                     (tl_w,"#a78bfa","Weighted Density","Units")]
            if tl_s: pd_list.append((tl_s,"#c026d3","Avg Speed","km/h"))
            for axx,(data,c,ttl,yl) in zip(axs,pd_list):
                axx.set_facecolor("#111827")
                axx.plot(range(len(data)),data,color=c,linewidth=2.2,marker="o",markersize=3)
                axx.fill_between(range(len(data)),data,alpha=0.15,color=c)
                avg_l=sum(data)/len(data)
                axx.axhline(avg_l,color=c,linestyle="--",linewidth=1,alpha=0.5,label=f"Avg:{avg_l:.1f}")
                if ttl=="Avg Speed":
                    axx.axhline(speed_limit,color="#f87171",linestyle=":",linewidth=1.5,label=f"Limit:{speed_limit}")
                axx.set_title(ttl,color="#64748b",fontsize=10,pad=8)
                axx.set_ylabel(yl,color="#475569",fontsize=8)
                axx.tick_params(colors="#475569",labelsize=8)
                axx.legend(facecolor="#1e293b",labelcolor="white",fontsize=8)
                for sp in axx.spines.values(): sp.set_color("#1e3a5f")
                axx.grid(color="#1e293b",linewidth=0.5)
            plt.tight_layout()
            bv=BytesIO()
            plt.savefig(bv,format="png",dpi=130,bbox_inches="tight",facecolor="#0d1117")
            st.image(bv,use_container_width=True)
            plt.close()

        if len(predictor.history)>=3:
            pn=predictor.predict_next(3); risk,rc=predictor.congestion_risk()
            st.markdown(
                '<div class="fcard"><p class="ftitle">TRAFFIC PREDICTION</p>'
                f'<p style="color:#94a3b8;font-size:12px;margin:0;">'
                f'Next 3: <span style="color:#60a5fa;font-weight:700;">{pn}</span> | '
                f'Risk: <span style="color:{rc};font-weight:700;">{risk}</span></p></div>',
            unsafe_allow_html=True)

        st.markdown("<br>",unsafe_allow_html=True)
        with open(out_path,"rb") as fo:
            st.download_button("Download Annotated Video",data=fo.read(),
                file_name=f"ai_traffic_{sim_mode.lower().replace(' ','_')}.mp4",
                mime="video/mp4",use_container_width=True)
        os.unlink(vid_path)

    else:
        st.markdown("""<div style="background:linear-gradient(135deg,#0f172a,#1e1b4b);border:2px dashed #312e81;border-radius:16px;padding:48px;text-align:center;">
        <p style="font-size:48px;margin:0;">🎬</p>
        <p style="font-family:Orbitron,monospace;color:#6366f1;font-size:16px;margin:12px 0 4px;">UPLOAD TRAFFIC VIDEO</p>
        <p style="color:#475569;font-size:12px;margin:0;">Speed | Wrong-Way | Tracking | Emergency | Prediction | Alerts | Scenarios</p>
        </div>""",unsafe_allow_html=True)

# ════════════════════════════════════
#  TAB 3 — MULTI-INTERSECTION
# ════════════════════════════════════
with tab3:
    st.markdown("<div class='sh'><p class='stitle'>Multi-Intersection Coordination - Junction A B C Green Wave</p></div>",unsafe_allow_html=True)

    st.markdown(
        '<div class="fcard"><p class="ftitle">HOW GREEN WAVE WORKS</p>'
        '<div style="display:flex;gap:14px;flex-wrap:wrap;">'
        '<div style="flex:1;background:rgba(99,102,241,.08);border:1px solid #312e81;border-radius:8px;padding:12px;">'
        '<p style="color:#4ade80;font-size:11px;font-weight:600;margin:0 0 4px;">Junction A — ACTIVE</p>'
        '<p style="color:#64748b;font-size:11px;margin:0;">Main intersection. AI controls based on real detection data.</p></div>'
        '<div style="flex:1;background:rgba(99,102,241,.08);border:1px solid #312e81;border-radius:8px;padding:12px;">'
        '<p style="color:#60a5fa;font-size:11px;font-weight:600;margin:0 0 4px;">Junction B — SYNCING</p>'
        '<p style="color:#64748b;font-size:11px;margin:0;">Pre-adjusts timing based on A flow. Offset = A green + 5s.</p></div>'
        '<div style="flex:1;background:rgba(99,102,241,.08);border:1px solid #312e81;border-radius:8px;padding:12px;">'
        '<p style="color:#a78bfa;font-size:11px;font-weight:600;margin:0 0 4px;">Junction C — SYNCED</p>'
        '<p style="color:#64748b;font-size:11px;margin:0;">Far downstream. Offset = A green x 2 + 10s for green wave.</p></div>'
        '</div></div>',
    unsafe_allow_html=True)

    st.markdown("<br>",unsafe_allow_html=True)

    # Use last history or defaults
    if st.session_state.history:
        last=st.session_state.history[-1]
        main_total=last["Vehicles"]
        main_counts={"car":max(0,main_total-2),"truck":1,"bus":1}
        main_gt=last["Green(s)"]
    else:
        main_total=8; main_counts={"car":6,"truck":1,"bus":1}
        main_gt=smart_green(8,main_counts,0,1.0,1.0)

    # Compute junction signals
    tf3,_=tod_factor(); _,wm3=auto_weather()
    junctions={}
    for jn,scale,offset in [("A",1.0,0),("B",0.7,main_gt+5),("C",0.4,main_gt*2+10)]:
        tot=max(0,int(main_total*scale+random.randint(-1,1)))
        cc={k:max(0,int(v*scale)) for k,v in main_counts.items()}
        wd3=weighted_density(cc)
        gt3=smart_green(tot,cc,0,wm3,tf3)
        dl3,dc3=density_label(tot)
        junctions[jn]={"total":tot,"counts":cc,"wd":wd3,"gt":gt3,
                       "density":dl3,"dens_col":dc3,"offset":offset,
                       "status":"Active" if jn=="A" else "Coordinating" if jn=="B" else "Synced"}

    # Green wave chart
    st.markdown("<div class='sh'><p class='stitle'>Green Wave Timeline</p></div>",unsafe_allow_html=True)
    fig_j,ax_j=plt.subplots(figsize=(16,4),facecolor="#0d1117")
    ax_j.set_facecolor("#111827")
    jcolors=["#4ade80","#60a5fa","#a78bfa"]
    for i,(jn,jc) in enumerate(zip(["A","B","C"],jcolors)):
        jd=junctions[jn]; off=jd["offset"]; gt_j=jd["gt"]
        ax_j.barh(i,gt_j,left=off,height=0.5,color=jc,alpha=0.85)
        ax_j.barh(i,3,left=off+gt_j,height=0.5,color="#fbbf24",alpha=0.7)
        ax_j.barh(i,15,left=off+gt_j+3,height=0.5,color="#f87171",alpha=0.4)
        ax_j.text(off+gt_j/2,i,"GREEN",ha="center",va="center",color="black",fontsize=9,fontweight="bold")
        ax_j.text(off-1,i,f"Junction {jn}\n({jd['total']}v, {jd['gt']}s)",
            ha="right",va="center",color=jc,fontsize=9,fontweight="bold")
    ax_j.set_xlabel("Time (seconds)",color="#475569",fontsize=9)
    ax_j.set_title("Green Wave Coordination - Junction A → B → C",color="#64748b",fontsize=11,pad=10)
    ax_j.tick_params(colors="#475569"); ax_j.set_yticks([]); ax_j.set_xlim(-25,130)
    for sp in ax_j.spines.values(): sp.set_color("#1e3a5f")
    ax_j.grid(axis="x",color="#1e293b",linewidth=0.5)
    plt.tight_layout()
    bj=BytesIO()
    plt.savefig(bj,format="png",dpi=130,bbox_inches="tight",facecolor="#0d1117")
    st.image(bj,use_container_width=True)
    plt.close()

    # Junction cards
    st.markdown("<div class='sh'><p class='stitle'>Junction Status</p></div>",unsafe_allow_html=True)
    j1,j2,j3=st.columns(3)
    dc_map3={"LOW":"#4ade80","MEDIUM":"#fbbf24","HIGH":"#fb923c","CRITICAL":"#f87171","EMPTY":"#94a3b8"}
    for col,(jn,jc) in zip([j1,j2,j3],zip(["A","B","C"],jcolors)):
        jd=junctions[jn]; is_act=(jn=="A")
        bg="linear-gradient(135deg,#0a1a0a,#0f2d0f)" if is_act else "rgba(0,0,0,.3)"
        border="2px solid #16a34a" if is_act else "1px solid #1e3a5f"
        with col:
            html  = f'<div style="background:{bg};border:{border};border-radius:12px;padding:14px;text-align:center;">'
            html += f'<p style="font-family:Orbitron,monospace;color:{jc};font-size:18px;font-weight:900;margin:0;">JUNCTION {jn}</p>'
            html += f'<p style="color:{jc};font-size:12px;font-weight:700;margin:4px 0;">{jd["status"].upper()}</p>'
            html += f'<hr style="border-color:#1e3a5f;margin:8px 0;">'
            html += f'<p style="font-family:Orbitron,monospace;color:{"#4ade80" if is_act else "#60a5fa"};font-size:22px;font-weight:900;margin:0;">{jd["gt"]}s GREEN</p>'
            html += f'<p style="color:#64748b;font-size:10px;margin:2px 0;">Offset: {jd["offset"]}s from A</p>'
            html += f'<hr style="border-color:#1e3a5f;margin:6px 0;">'
            html += f'<p style="color:{dc_map3.get(jd["density"],"#94a3b8")};font-size:12px;font-weight:700;margin:2px 0;">{jd["density"]}</p>'
            html += f'<p style="color:#64748b;font-size:10px;margin:1px 0;">{jd["total"]} vehicles | {jd["wd"]:.1f} weighted</p>'
            html += f'<p style="color:#64748b;font-size:10px;margin:1px 0;">{"Main controller" if jn=="A" else "Downstream sync" if jn=="B" else "Far downstream"}</p>'
            html += '</div>'
            st.markdown(html,unsafe_allow_html=True)

    # Network metrics
    st.markdown("<br>",unsafe_allow_html=True)
    n1,n2,n3,n4=st.columns(4)
    total_net=sum(j["total"] for j in junctions.values())
    avg_gt_net=round(sum(j["gt"] for j in junctions.values())/3,1)
    for col,lbl,val,ch in [
        (n1,"Network Vehicles",str(total_net),"#4ade80"),
        (n2,"Avg Green",f"{avg_gt_net}s","#60a5fa"),
        (n3,"Wave Offset A-B",f"{junctions['B']['offset']}s","#a78bfa"),
        (n4,"Junctions Active","3","#fbbf24"),
    ]:
        with col:
            st.markdown(f'<div class="mcard"><div class="mval" style="color:{ch};font-size:20px;">{val}</div><div class="mlbl">{lbl}</div></div>',unsafe_allow_html=True)

# ════════════════════════════════════
#  TAB 4 — ANALYTICS & PDF
# ════════════════════════════════════
with tab4:
    st.markdown("<div class='sh'><p class='stitle'>AI vs Traditional Comparison + Analytics + PDF Report</p></div>",unsafe_allow_html=True)

    sc_data=[{"label":"Empty (0v)","n":0},{"label":"Low (3v)","n":3},
             {"label":"Medium (8v)","n":8},{"label":"Medium (12v)","n":12},
             {"label":"High (18v)","n":18},{"label":"Critical (25v)","n":25}]
    ai_gs=[smart_green(s["n"],{"car":s["n"]},0,1.0,1.0) for s in sc_data]
    t_ws=[90-30+s["n"]*1.5 for s in sc_data]
    a_ws=[max(0,90-g+s["n"]*0.8) for g,s in zip(ai_gs,sc_data)]
    imps=[round((t-a)/max(t,1)*100,1) for t,a in zip(t_ws,a_ws)]
    avg_imp=round(sum(imps)/len(imps),1)
    scans=len(st.session_state.trad_waits)
    real_imp=0
    if scans>0:
        real_imp=round((sum(st.session_state.trad_waits)-sum(st.session_state.ai_waits))/max(sum(st.session_state.trad_waits),1)*100,1)

    s1,s2,s3,s4=st.columns(4)
    with s1:
        st.markdown(
            '<div style="background:linear-gradient(135deg,#0a1a0a,#0d2d1a);border:1px solid #16a34a;border-radius:12px;padding:18px;text-align:center;">'
            '<p style="color:#475569;font-size:9px;letter-spacing:2px;margin:0 0 4px;">AVG IMPROVEMENT</p>'
            f'<p style="font-family:Orbitron,monospace;color:#4ade80;font-size:32px;font-weight:900;margin:0;">{avg_imp}%</p>'
            '<p style="color:#64748b;font-size:10px;margin:4px 0 0;">vs Fixed 30s</p></div>',
        unsafe_allow_html=True)
    with s2:
        st.markdown('<div class="mcard"><div class="mval" style="color:#f87171;">30s</div><div class="mlbl">TRADITIONAL FIXED</div></div>',unsafe_allow_html=True)
    with s3:
        st.markdown('<div class="mcard"><div class="mval" style="color:#4ade80;">10-60s</div><div class="mlbl">AI ADAPTIVE</div></div>',unsafe_allow_html=True)
    with s4:
        if scans>0:
            st.markdown(
                '<div style="background:linear-gradient(135deg,#0a1a0a,#0d2d1a);border:1px solid #16a34a;border-radius:12px;padding:18px;text-align:center;">'
                '<p style="color:#475569;font-size:9px;letter-spacing:2px;margin:0 0 4px;">YOUR SESSION</p>'
                f'<p style="font-family:Orbitron,monospace;color:#4ade80;font-size:32px;font-weight:900;margin:0;">{real_imp}%</p>'
                f'<p style="color:#64748b;font-size:10px;margin:4px 0 0;">from {scans} scans</p></div>',
            unsafe_allow_html=True)
        else:
            st.markdown('<div class="mcard"><div class="mval" style="color:#64748b;">--</div><div class="mlbl">RUN DETECTIONS FIRST</div></div>',unsafe_allow_html=True)

    st.markdown("<br>",unsafe_allow_html=True)

    # Charts
    fig_c,(ca1,ca2,ca3)=plt.subplots(1,3,figsize=(20,6),facecolor="#0d1117")
    labels=[s["label"] for s in sc_data]; x=np.arange(len(labels)); w=0.35
    for axx in [ca1,ca2,ca3]: axx.set_facecolor("#111827")
    b1=ca1.bar(x-w/2,t_ws,w,label="Fixed 30s",color="#f87171",alpha=0.85,edgecolor="#1e293b")
    b2=ca1.bar(x+w/2,a_ws,w,label="AI Adaptive",color="#4ade80",alpha=0.85,edgecolor="#1e293b")
    for bar in b1: ca1.text(bar.get_x()+bar.get_width()/2,bar.get_height()+.5,f"{bar.get_height():.0f}s",ha="center",va="bottom",color="#f87171",fontsize=8,fontweight="bold")
    for bar in b2: ca1.text(bar.get_x()+bar.get_width()/2,bar.get_height()+.5,f"{bar.get_height():.0f}s",ha="center",va="bottom",color="#4ade80",fontsize=8,fontweight="bold")
    ca1.set_title("Average Waiting Time (s)",color="#94a3b8",fontsize=11,pad=10)
    ca1.set_xticks(x); ca1.set_xticklabels(labels,color="#475569",fontsize=8,rotation=15)
    ca1.tick_params(colors="#475569"); ca1.legend(facecolor="#1e293b",labelcolor="white",fontsize=9)
    for sp in ca1.spines.values(): sp.set_color("#1e3a5f")
    for i,(lbl,vi) in enumerate(zip(labels,imps)):
        cc="#4ade80" if vi>=0 else "#f87171"
        ca2.bar(i,vi,color=cc,alpha=0.85,edgecolor="#1e293b")
        ca2.text(i,vi+0.5,f"{vi}%",ha="center",va="bottom",color="white",fontsize=9,fontweight="bold")
    ca2.axhline(0,color="#555",linewidth=1)
    ca2.set_title("AI Improvement over Fixed (%)",color="#94a3b8",fontsize=11,pad=10)
    ca2.set_xticks(range(len(labels))); ca2.set_xticklabels(labels,color="#475569",fontsize=8,rotation=15)
    ca2.tick_params(colors="#475569")
    for sp in ca2.spines.values(): sp.set_color("#1e3a5f")
    cnts=[s["n"] for s in sc_data]
    ca3.plot(cnts,[30]*len(cnts),color="#f87171",linewidth=2.5,linestyle="--",label="Fixed 30s",marker="s",markersize=7)
    ca3.plot(cnts,ai_gs,color="#4ade80",linewidth=2.5,linestyle="-",label="AI Adaptive",marker="o",markersize=8)
    ca3.fill_between(cnts,[30]*len(cnts),ai_gs,alpha=0.12,color="#4ade80")
    ca3.set_title("Green Phase vs Vehicle Count",color="#94a3b8",fontsize=11,pad=10)
    ca3.set_xlabel("Vehicle Count",color="#475569",fontsize=9)
    ca3.set_ylabel("Green Time (s)",color="#475569",fontsize=9)
    ca3.tick_params(colors="#475569"); ca3.legend(facecolor="#1e293b",labelcolor="white",fontsize=9)
    for sp in ca3.spines.values(): sp.set_color("#1e3a5f")
    plt.tight_layout()
    bc=BytesIO()
    plt.savefig(bc,format="png",dpi=140,bbox_inches="tight",facecolor="#0d1117")
    st.image(bc,use_container_width=True)
    plt.close()

    # Feature table
    st.markdown("<br>",unsafe_allow_html=True)
    st.markdown("<div class='sh'><p class='stitle'>Complete Feature Comparison</p></div>",unsafe_allow_html=True)
    comp_df=pd.DataFrame({
        "Feature":["Signal Timing","Vehicle Weighting","Waiting Time","Emergency","Incident",
                   "Weather","Time-of-Day","Starvation","Fail-Safe","Tracking",
                   "Priority Score","Speed Estimation","Traffic Prediction",
                   "Wrong-Way","Pedestrian","Multi-Junction","PDF Report"],
        "Traditional":["Fixed 30s","Not used","Not tracked","Manual","Not detected",
                       "Not used","Not used","No","No","No","Count only",
                       "No","No","No","No","Single junction","None"],
        "This AI System":["Dynamic 10-60s","Truck=3x Bus=2.5x Van=1.5x Car=1x","Tracked + formula",
                          "Auto 60s override","0 vehicles or 2+ trucks",
                          "Rain=1.3x Night=1.2x","Peak=1.5x Lunch=1.2x Night=0.7x",
                          "Force green after 120s","Fallback 30s fixed",
                          "Centroid tracker unique IDs","(wd x 2)+(wait x 1.5)+emg bonus",
                          "Pixel displacement per vehicle","Linear regression next 3 steps",
                          "Direction vector tracking","School hours yellow extension",
                          "A-B-C green wave offset","Auto PDF with history + speed"],
    })
    st.dataframe(comp_df,use_container_width=True,hide_index=True)

    # History
    if st.session_state.history:
        st.markdown("<br>",unsafe_allow_html=True)
        st.markdown("<div class='sh'><p class='stitle'>Detection History</p></div>",unsafe_allow_html=True)
        hist=st.session_state.history
        avg_v=round(sum(h["Vehicles"] for h in hist)/len(hist),1)
        avg_g=round(sum(h["Green(s)"] for h in hist)/len(hist),1)
        avg_s2=round(sum(h["Saved(s)"] for h in hist)/len(hist),1)
        emg_c=sum(1 for h in hist if h["Emergency"]=="YES")
        h1,h2,h3,h4=st.columns(4)
        for col,lbl,val,ch in [
            (h1,"Avg Vehicles",str(avg_v),"#4ade80"),
            (h2,"Avg Green",f"{avg_g}s","#60a5fa"),
            (h3,"Avg Saved",f"{avg_s2}s","#4ade80"),
            (h4,"Emergencies",str(emg_c),"#ff4444"),
        ]:
            with col:
                st.markdown(f'<div class="mcard"><div class="mval" style="color:{ch};font-size:20px;">{val}</div><div class="mlbl">{lbl}</div></div>',unsafe_allow_html=True)

        if len(hist)>=3:
            last3=[h["Vehicles"] for h in hist[-3:]]
            trend=last3[-1]-last3[0]
            pc="#f87171" if trend>0 else "#4ade80" if trend<0 else "#fbbf24"
            pd_="INCREASING" if trend>0 else "DECREASING" if trend<0 else "STABLE"
            rec="Extend green" if trend>2 else "Reduce green" if trend<-2 else "Maintain"
            st.markdown(
                '<div class="fcard"><p class="ftitle">TRAFFIC PREDICTION</p>'
                f'<p style="color:#94a3b8;font-size:12px;margin:0;">'
                f'Trend: <span style="color:{pc};font-weight:700;">{pd_}</span> | '
                f'Predicted next: ~{max(0,last3[-1]+trend)} vehicles | '
                f'Recommendation: <span style="color:#6366f1;font-weight:700;">{rec}</span>'
                f'</p></div>',
            unsafe_allow_html=True)

        st.markdown("<br>",unsafe_allow_html=True)
        df=pd.DataFrame(hist[-15:])
        st.dataframe(df,use_container_width=True,hide_index=True)

        cd1,cd2,cd3=st.columns(3)
        with cd1:
            csv=pd.DataFrame(hist).to_csv(index=False).encode()
            st.download_button("Export CSV",data=csv,
                file_name="ai_traffic_report.csv",mime="text/csv",use_container_width=True)
        with cd2:
            if PDF_OK:
                metrics_pdf={
                    "Total Detections":len(hist),"Avg Vehicles":avg_v,
                    "Avg Green":f"{avg_g}s","Avg Saved":f"{avg_s2}s",
                    "Emergency Events":emg_c,"AI Improvement":f"{real_imp}%" if scans>0 else "N/A",
                }
                pdf_b=make_pdf(hist,st.session_state.speed_hist,metrics_pdf)
                if pdf_b:
                    st.download_button("Export PDF Report",data=pdf_b,
                        file_name="ai_traffic_report.pdf",mime="application/pdf",
                        use_container_width=True)
            else:
                st.info("Run: !pip install reportlab -q  then restart")
        with cd3:
            if st.button("Clear All History",use_container_width=True):
                st.session_state.history=[]
                st.session_state.trad_waits=[]
                st.session_state.ai_waits=[]
                st.session_state.speed_hist=[]
                st.rerun()

# FOOTER
st.markdown("""
<div class="footer">
<p><span>Sugandha Srivastava</span> &nbsp; <span>Sudhanshu Shukla</span> &nbsp; <span>Sakshi Chaturvedi</span> &nbsp; <span>Shreya Dutta</span></p>
<p>Dept. of Computer Science and Engineering &nbsp; Nitra Technical Campus, Ghaziabad &nbsp; Guide: <span>Dr. Rajesh Kumar</span></p>
</div>
""", unsafe_allow_html=True)

Writing app.py


In [4]:
!pip install reportlab -q

import subprocess, time, re

subprocess.run(["pkill","-f","streamlit"],   capture_output=True)
subprocess.run(["pkill","-f","cloudflared"], capture_output=True)
time.sleep(3)

subprocess.Popen(
    ["streamlit","run","app.py",
     "--server.port","8501",
     "--server.headless","true",
     "--server.enableCORS","false"],
    stdout=open("/content/logs.txt","w"),
    stderr=subprocess.STDOUT
)
time.sleep(12)

proc=subprocess.Popen(
    ["./cloudflared-linux-amd64","tunnel","--url","http://localhost:8501"],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, universal_newlines=True
)
for line in proc.stdout:
    if "trycloudflare.com" in line:
        url=re.search(r'https://\S+\.trycloudflare\.com',line)
        if url:
            print("="*55)
            print("OPEN THIS URL:", url.group())
            print("="*55)
            break

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 7.0 MB/s eta 0:00:00
OPEN THIS URL: https://gray-punch-newton-killing.trycloudflare.com
